# NSL-KDD - Análise exploratória e diagnóstico inicial

Etapa 1, com entrega em 01/11 segundo o enunciado atualizado. Usa exclusivamente o treino oficial do Canvas e não treina modelos nem aplica limpeza.

O alvo oficial é `class`, com `normal` e `anomaly`. Os CSVs fornecidos têm os 41 atributos originais e oito atributos adicionais, totalizando 49 entradas. O campo `difficulty` citado na descrição geral não está nesses CSVs; a caracterização abaixo usa a estrutura efetivamente fornecida.

As estatísticas são calculadas a cada execução. O teste sem rótulos será utilizado para gerar a submissão somente depois de definir o pipeline final com o treino.

## 1. Carregamento reprodutível

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from IPython.display import Markdown, display
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
RAIZ=Path.cwd()
if not (RAIZ / "preprocessamento.py").is_file(): raise FileNotFoundError("Abra na raiz do projeto.")
sys.path.insert(0,str(RAIZ))
from preprocessamento import *
DATA_PATH=RAIZ / "data" / "raw" / "treino.csv"
dados=carregar_treino(DATA_PATH)
print(DATA_PATH.name, dados.shape)

## 2. Caracterização e desbalanceamento

In [ ]:
numericas,categoricas=identificar_colunas_por_tipo(dados)
perfil=pd.DataFrame({"tipo":dados.dtypes.astype(str),"ausencias":dados.isna().sum(),"distintos":dados.nunique(dropna=False)})
display(perfil)
classe=dados[ALVO].value_counts().rename("frequencia").to_frame(); classe["percentual"]=(100*classe.frequencia/len(dados)).round(3)
display(classe)
sns.countplot(data=dados,x=ALVO,order=classe.index,hue=ALVO,legend=False); plt.title("Distribuição de class"); plt.show()

A predominância de `normal` caracteriza desbalanceamento. Nenhum balanceamento é aplicado agora; ele será comparado em validação na Etapa 2/3.

### Atributos constantes e decisão já aprovada

A tabela abaixo identifica constantes do treino. Foi aprovada em 14/10 a retirada de `num_outbound_cmds` das entradas dos futuros modelos, por ser sempre zero. A constância é reconferida na base oficial. A análise exploratória preserva essa coluna para documentar a justificativa; não executa agora uma seleção de atributos.

In [ ]:
constantes = [c for c in dados.columns if c != ALVO and dados[c].nunique(dropna=False) == 1]
display(pd.DataFrame({"atributo_constante": constantes, "valor": [dados[c].iloc[0] for c in constantes]}))

## 3. Investigação do atributo `su_attempted`

In [ ]:
su=investigar_su_attempted(dados)
display(su["frequencias"]); display(su["por_classe"]); display(su["por_servico"]); display(su["por_root_shell"])
print("Registros com valor 2:",len(su["registros_com_valor_2"])); print("Duplicatas nesse subconjunto:",su["duplicatas_valor_2"])

A definição binária original de `su_attempted` prevê 0 e 1. O valor 2 deve ser investigado como incompatível com essa definição, sem concluir sua causa ou recodificá-lo automaticamente. As alternativas para uma futura comparação incluem categoria especial, ausência com imputação ajustada só no treino, ou retirada dos registros afetados. A tabela por classe documenta o impacto antes de qualquer escolha.

## 4. Regras de domínio e faixas para investigação

Duração e bytes negativos e frações fora de [0, 1] são verificados separadamente das ausências. Os contadores também são comparados com referências indicativas herdadas da análise anterior. Sem documentação do gerador que estabeleça seus máximos, exceder 511 ou 255 não basta para concluir erro: pode haver diferença de escala ou outra definição. Essas referências não são critérios de remoção.

In [ ]:
print("Taxas verificadas em [0, 1]:", identificar_taxas(dados))
display(analisar_violacoes_dominio(dados))
resumo_taxas = resumir_taxas_invalidas(dados)
print("Registros com ao menos uma taxa fora da faixa:", resumo_taxas["registros_com_taxa_invalida"],
      f"({resumo_taxas['percentual_base']}% da base)")
display(exemplos_violacoes(dados))
print("Referências indicativas dos contadores, sem conclusão automática de erro:")
display(analisar_faixas_referencia(dados))

## 5. Entradas nominais, variações de caixa e extremos

`protocol_type`, `service` e `flag` são categorias sem ordem. `protocolo_cod` também é nominal: seus números identificam protocolos, sem representar magnitude ou distância. O perfil por dtype descreve o armazenamento; na análise de associação com a classe, `protocolo_cod` é tratado por V de Cramér. `tipo_alerta_soc` é analisado como categoria e está provisoriamente excluído dos futuros experimentos.

Nesta Etapa 1, identificamos categorias, cardinalidade, ausências e variantes de caixa/espaços. A função abaixo compara versões normalizadas em séries temporárias; ela não altera os dados. Ainda não há um codificador de entradas ajustado ou aplicado.

Para a Etapa 2, que exige pelo menos uma técnica de conversão simbólico-numérica justificada, a proposta inicial é one-hot encoding: representar cada categoria por um indicador numérico 0/1, sem impor ordem. A escolha ainda está em discussão. É necessário definir a padronização das categorias e as políticas para ausências e categorias novas, aprender o vocabulário apenas na parcela de treino de cada rodada e reutilizar o mesmo esquema na validação. O teste será transformado apenas pelo pipeline final ajustado com o treino oficial. Avaliar o aumento de dimensionalidade, especialmente de `service`. Se mantido em uma representação, `protocolo_cod` também deve receber tratamento nominal.

Referência da proposta: [codificação categórica no scikit-learn](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features).

In [ ]:
display(encontrar_inconsistencias_caixa(dados,categoricas))
interesse=[x for x in ["duration","src_bytes","dst_bytes","num_root","num_compromised"] if x in dados]
display(dados[interesse].describe(percentiles=[.01,.05,.5,.95,.99]).T)
for coluna in interesse:
 validos=dados.loc[dados[coluna].ge(0),coluna].dropna()
 fig,ax=plt.subplots(1,2,figsize=(12,3)); sns.boxplot(x=dados[coluna],ax=ax[0]); ax[0].set_title(coluna); sns.histplot(np.log1p(validos),bins=40,ax=ax[1]); ax[1].set_title(f"log1p({coluna}) - apenas valores válidos"); plt.show()

Nem todo extremo é erro: evento raro pode ser relevante para intrusão. Nenhum outlier é removido nesta etapa. Para a visualização logarítmica, somente valores não negativos são exibidos; essa filtragem é exclusivamente gráfica e não altera os dados originais.

## 6. Redundâncias, duplicatas e conflitos

Duplicatas completas incluem a classe. Repetições de entradas desconsideram a classe e podem incluir tanto duplicatas coerentes quanto conflitos. A terceira contagem mostra todos os registros dos grupos com rótulos diferentes. Essas métricas se sobrepõem e não devem ser somadas.

A igualdade inclui os 49 atributos e o mesmo padrão de ausência. Isso caracteriza registros equivalentes nos campos observados; não prova, por si só, que representam o mesmo evento real. Os índices originais são preservados nos exemplos.

In [ ]:
display(verificar_redundancias(dados))
resumo, conflitos = analisar_duplicatas(dados)
display(resumo)
display(conflitos.head())
impacto = impacto_dos_conflitos(dados, conflitos)
print("Efeito hipotético de retirar todos os registros conflitantes, sem aplicar exclusão:")
display(impacto)

## 7. Investigação de possíveis vazamentos

O enunciado pede investigar cada atributo em relação à classe antes de treinar modelos. Aqui isso é feito com medidas adequadas ao tipo de variável:

- **Pearson** para atributos numéricos, com `anomaly=1` e `normal=0`. Nesse caso equivale à correlação ponto-bisserial. São usados apenas os pares observados e o número de pares é informado. Não são atribuídos zeros aos valores ausentes.
- **V de Cramér** para nominais, incluindo `protocolo_cod`. O código numérico do protocolo não cria uma ordem entre categorias. A ausência é um nível explícito nessa medida, para também investigar seu padrão.
- Para uma constante, a correlação fica indefinida. Os valores são mantidos como ausentes na tabela, em vez de apresentados como correlação zero.

As duas medidas têm interpretações diferentes. Pearson descreve relação linear; o V de Cramér descreve associação entre categorias. Correlação alta não demonstra vazamento, e correlação baixa não elimina utilidade não linear. As tabelas são diagnósticas: não aplicam um filtro de atributos nem indicam um modelo vencedor.

In [ ]:
associacoes = analisar_associacoes_com_classe(dados)
associacoes_numericas = associacoes.loc[associacoes["medida"].eq("Pearson")].copy()
associacoes_numericas["magnitude"] = associacoes_numericas["valor"].abs()
associacoes_numericas = associacoes_numericas.sort_values("magnitude", ascending=False)
print("Todos os atributos numéricos; pares observados por atributo:")
display(associacoes_numericas.drop(columns="magnitude"))
print("Todos os atributos nominais; ausência considerada como categoria:")
display(associacoes.loc[associacoes["medida"].eq("Cramer V")])

top_numericos = associacoes_numericas.dropna(subset=["valor"]).head(12).sort_values("valor")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_numericos["atributo"], top_numericos["valor"], color="#35658a")
ax.axvline(0, color="black", linewidth=.8)
ax.set_xlim(-1, 1)
ax.set_xlabel("Pearson com anomaly=1 e normal=0")
ax.set_title("Maiores magnitudes de correlação linear no treino")
fig.tight_layout()
plt.show()

### 7.1 Evidências das três colunas sob investigação temporal

A suspeita depende da origem da informação e de ela estar disponível ao classificar a conexão. Não basta que uma coluna esteja presente tanto no treino quanto no teste. Também é necessário verificar como ela foi produzida.

Além dos valores, examinamos o padrão de ausência por classe. Imputar um atributo não elimina um eventual vazamento de origem; um indicador de ausência também pode conservar informações de uma ação posterior.

In [ ]:
suspeitas_presentes = [c for c in COLUNAS_SUSPEITAS_LEAKAGE if c in dados]
display(associacoes.loc[associacoes["atributo"].isin(suspeitas_presentes)])
ausencias_por_classe = resumir_ausencias_por_classe(dados, suspeitas_presentes)
display(ausencias_por_classe)

campos_numericos_suspeitos = [c for c in ["score_ids_legado", "tempo_resposta_analista_min"] if c in dados]
if campos_numericos_suspeitos:
    display(dados.groupby(ALVO)[campos_numericos_suspeitos].agg(["count", "min", "median", "mean", "max"]))
if "tipo_alerta_soc" in dados:
    print("Alertas por classe, em contagens:")
    display(pd.crosstab(dados["tipo_alerta_soc"].fillna("<AUSENTE>"), dados[ALVO]))
    print("Percentuais dentro de cada classe:")
    display(pd.crosstab(dados["tipo_alerta_soc"].fillna("<AUSENTE>"), dados[ALVO], normalize="columns").mul(100).round(2))

if "tempo_resposta_analista_min" in dados:
    faltas_tempo = ausencias_por_classe.loc[ausencias_por_classe["atributo"].eq("tempo_resposta_analista_min")]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(faltas_tempo["classe"], faltas_tempo["percentual_ausente"], color="#35658a")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Registros com valor ausente (%)")
    ax.set_title("Ausência do tempo de resposta por classe")
    fig.tight_layout()
    plt.show()

if "score_ids_legado" in dados:
    fig, ax = plt.subplots(figsize=(8, 4))
    for rotulo, cor in [("normal", "#35658a"), ("anomaly", "#b25a38")]:
        valores = dados.loc[dados[ALVO].eq(rotulo), "score_ids_legado"].dropna()
        ax.hist(valores, bins=np.linspace(0, 1, 31), density=True, histtype="step", linewidth=1.6, label=rotulo, color=cor)
    ax.set_xlabel("score_ids_legado")
    ax.set_ylabel("Densidade dentro da classe")
    ax.set_title("Distribuição do escore legado por classe")
    ax.legend()
    fig.tight_layout()
    plt.show()

### 7.2 Interpretação e limites das conclusões

| Atributo | Evidência a conferir nas tabelas | Questão de origem e disponibilidade |
| --- | --- | --- |
| `score_ids_legado` | Escore associado à classe, com sobreposição das distribuições | Foi gerado por um sistema anterior, disponível no momento da previsão, ou calculado usando rótulos que queremos prever? |
| `tipo_alerta_soc` | Associação forte, mas existem alertas em `normal` e registros `anomaly` com `OK` | O alerta é produzido antes da classificação ou depois da investigação/triagem? |
| `tempo_resposta_analista_min` | Ausência muito diferente entre as classes, além da diferença dos valores observados | O nome sugere uma resposta posterior do analista. Confirmar quando o valor passa a existir. |

O enunciado não fornece o processo de geração dessas colunas. Portanto, a relação estatística está demonstrada, mas a origem temporal precisa de confirmação. Nesta etapa nenhuma das três colunas é removida. Um campo indisponível no momento da previsão deve ser excluído das entradas futuras com essa justificativa documentada. A avaliação por validação cruzada, sozinha, não corrige a presença de informação posterior ao evento.

Para a etapa de experimentos, manter uma referência sem exclusão de conflitos e comparar tratamentos. A nova base exige cuidado especial com a perda relativa da classe `anomaly`. Duplicatas também precisam ser consideradas ao construir os grupos de validação, para evitar que cópias apareçam no treino e na validação da mesma rodada.

## 8. Síntese e diagnóstico final

As contagens abaixo são recalculadas com o treino oficial. O diagnóstico breve reúne os principais achados e seus limites, sem transformar associações em prova de vazamento nem inconsistências em critérios automáticos de exclusão. Nenhum desses resultados representa F1 ou avaliação de um classificador.

In [ ]:
resumo_classes = dados[ALVO].value_counts(normalize=True).mul(100).round(2)
print(f"Base oficial: {len(dados):,} registros, {dados.shape[1] - 1} atributos e alvo class.")
print(f"Distribuição: normal={resumo_classes.get('normal', 0)}%; anomaly={resumo_classes.get('anomaly', 0)}%.")
print("Atributos com ausências:", int(dados.drop(columns=ALVO).isna().any().sum()))
print("Atributos constantes:", constantes)
print("Regras verificadas com ocorrências:")
display(analisar_violacoes_dominio(dados).query("violacoes > 0")[["atributo", "regra", "violacoes"]])
print("Variantes de caixa/espaços identificadas:", len(encontrar_inconsistencias_caixa(dados, categoricas)))
print("Relações determinísticas verificadas:", verificar_redundancias(dados).query("redundancia_comprovada")["atributo"].tolist())
display(resumo)
display(impacto)
print("Os limites indicativos dos contadores exigem investigação de escala e documentação; não são critérios automáticos de exclusão.")
print("A retirada futura de num_outbound_cmds foi aprovada; o original permanece no diagnóstico.")
print("A investigação temporal dos três sinais adicionais permanece condicionada à informação sobre sua geração.")
print("Na Etapa 2, comparar os tratamentos dentro do treino, preservando uma referência e medindo a perda por classe.")

# Diagnóstico breve para a entrega. Todas as contagens vêm das tabelas anteriores.
def formatar_numero(valor, casas=0):
    return format(valor, f",.{casas}f").translate(str.maketrans(",.", ".,"))

contagem_classes = dados[ALVO].value_counts()
ausencias = dados.drop(columns=ALVO).isna().sum()
regras = analisar_violacoes_dominio(dados).set_index("atributo")
resumo_por_tipo = resumo.set_index("tipo")["quantidade"]
impacto_por_classe = impacto.set_index("classe")
nominais = ["protocol_type", "service", "flag"]
cardinalidades_normalizadas = {
    c: dados[c].astype("string").str.strip().str.casefold().nunique(dropna=True)
    for c in nominais
}
grupos_conflitantes = conflitos.drop_duplicates(subset=[c for c in dados if c != ALVO]).shape[0]
faltas_tempo_por_classe = ausencias_por_classe.loc[
    ausencias_por_classe["atributo"].eq("tempo_resposta_analista_min")
].set_index("classe")["percentual_ausente"]

diagnostico_final = f"""
### Diagnóstico breve

**Base e classes.** O treino oficial contém {formatar_numero(len(dados))} registros e {dados.shape[1] - 1} atributos de entrada: os 41 originais e oito adicionais. Não há `difficulty` no CSV. São {formatar_numero(contagem_classes['normal'])} registros `normal` ({formatar_numero(resumo_classes['normal'], 2)}%) e {formatar_numero(contagem_classes['anomaly'])} `anomaly` ({formatar_numero(resumo_classes['anomaly'], 2)}%). O desbalanceamento exige acompanhar a classe minoritária; a acurácia isolada não será um critério suficiente na avaliação posterior.

**Ausências e inconsistências.** {int(ausencias.gt(0).sum())} atributos têm ausências; `tempo_resposta_analista_min` está ausente em {formatar_numero(100 * ausencias['tempo_resposta_analista_min'] / len(dados), 2)}% das linhas. Há {formatar_numero(regras.loc['duration', 'violacoes'])} durações negativas, {formatar_numero(regras.loc['src_bytes', 'violacoes'])} valores negativos em `src_bytes` e {formatar_numero(resumo_taxas['registros_com_taxa_invalida'])} registros com ao menos uma fração fora de [0, 1]. Esses conjuntos podem se sobrepor. O valor 2 de `su_attempted`, observado em {formatar_numero(regras.loc['su_attempted', 'violacoes'])} registros, precisa de esclarecimento antes de qualquer recodificação. Os limites indicativos dos contadores não comprovam erro; extremos também podem representar eventos legítimos.

**Entradas nominais.** `protocol_type`, `service` e `flag` não têm ordem. Uma comparação temporária, ignorando caixa e espaços laterais, encontra respectivamente {cardinalidades_normalizadas['protocol_type']}, {cardinalidades_normalizadas['service']} e {cardinalidades_normalizadas['flag']} categorias observadas, sem contar ausências. As variantes não foram substituídas na base. `protocolo_cod` continua nominal apesar do armazenamento numérico. A escolha e a aplicação da codificação ficam para a Etapa 2.

**Redundâncias e conflitos.** `num_outbound_cmds` é constante. As relações de `src_bytes_kb`, `dst_bytes_kb`, `total_bytes` e `same_srv_pct` foram confirmadas nas linhas comparáveis; `protocolo_cod` recodifica o protocolo. Há {formatar_numero(resumo_por_tipo['duplicatas exatas (inclui class)'])} repetições excedentes de linhas completas e {formatar_numero(len(conflitos))} registros em {formatar_numero(grupos_conflitantes)} grupos de entradas iguais com classes diferentes. As contagens de duplicatas e conflitos se sobrepõem. Retirar todos os conflitos excluiria {formatar_numero(impacto_por_classe.loc['anomaly', 'percentual_da_classe_envolvido'], 2)}% dos exemplos `anomaly`; isso impede adotar essa exclusão automaticamente. Igualdade dos campos observados e do padrão de ausência não prova que se trata do mesmo evento real.

**Possível vazamento e limites.** As associações de `score_ids_legado`, `tipo_alerta_soc` e `tempo_resposta_analista_min` com a classe exigem verificar sua origem e disponibilidade no momento da detecção. A ausência do tempo de resposta difere entre `normal` ({formatar_numero(faltas_tempo_por_classe['normal'], 2)}%) e `anomaly` ({formatar_numero(faltas_tempo_por_classe['anomaly'], 2)}%). Essas evidências não demonstram, sozinhas, vazamento: o processo de geração não foi documentado. As três colunas ficam provisoriamente fora dos experimentos futuros, preservadas nesta análise.

**Alcance desta etapa.** Os resultados descrevem apenas o treino oficial. Não foram aplicados imputação, codificação de entradas, balanceamento, remoção de registros ou treinamento de classificadores. O teste não foi utilizado na análise. Os achados orientam as comparações futuras; ainda não há evidência para escolher um tratamento ou modelo vencedor.
"""
display(Markdown(diagnostico_final))


## 9. Referências e reprodução

- Enunciado atualizado: *Enunciado_Trabalho_Competicao_IA_23-10-26-1.pdf*, fornecido pela disciplina.
- Dados oficiais: `data/raw/treino.csv`. O notebook não procura rótulos externos nem lê os atributos do teste.
- [Definições originais dos atributos do KDD](https://kdd.ics.uci.edu/databases/kddcup99/task.html).
- [Associação entre variáveis nominais, SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.association.html). A fórmula do V de Cramér é implementada com NumPy e conferida com o exemplo publicado.
- [Prevenção de vazamento, scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html).
- [Validação estratificada com grupos, scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html).

O relatório final deve declarar as ferramentas de apoio e suas finalidades, conforme o item de integridade acadêmica do enunciado.